# Notebook 5 — Batch Classification (Stretch Goal)

Run classification on all three sample images at once and display a results gallery.
This notebook is optional — complete it if you finish the main lab early.

> All images come from the git repo — no internet access required.

In [ ]:
# ── Cell 0: Sync lab materials from GitHub ────────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
# ── Configuration — auto-detected from namespace ──────────────────────────────
NAMESPACE      = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()
# Inference URL auto-detected from the active route
try:
    import subprocess
    _route = subprocess.check_output(
        ['oc', 'get', 'route', 'defect-classifier-ext', '-n', NAMESPACE,
         '-o', 'jsonpath={.spec.host}'], text=True).strip()
    INFERENCE_URL  = f'https://{_route}'
except Exception:
    _domain = os.environ.get('CLUSTER_DOMAIN', 'apps.itz-t53413.hub01-lb.techzone.ibm.com')
    INFERENCE_URL  = f'https://defect-classifier-ext-{NAMESPACE}.{_domain}'
MODEL_NAME     = 'defect-classifier'
IMG_SIZE       = 96
CLASSES        = ['contamination', 'crack', 'pass', 'scratch']
INFER_ENDPOINT = f'{INFERENCE_URL}/v2/models/{MODEL_NAME}/infer'
print(f'Namespace     : {NAMESPACE}')
print(f'Inference URL : {INFERENCE_URL}')

In [ ]:
import numpy as np, requests, os
from PIL import Image
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Local sample images from git repo — no internet required ──────────────────
SAMPLE_DIR  = str(pathlib.Path(LAB) / 'sample-images')
TEST_IMAGES = {
    'Pass (clean PCB)'         : f'{SAMPLE_DIR}/pass.png',
    'Scratch defect'           : f'{SAMPLE_DIR}/scratch.jpg',
    'Crack defect'             : f'{SAMPLE_DIR}/crack.jpg',
}

# Verify all images exist
for label, path in TEST_IMAGES.items():
    status = '✅' if os.path.exists(path) else '❌ MISSING'
    print(f'  {status}  {label}')

def preprocess(img_path):
    img = Image.open(img_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0
    arr = np.transpose(arr, (2, 0, 1))
    return np.expand_dims(arr, axis=0)

def classify(arr):
    payload = {'inputs': [{
        'name': 'images', 'shape': list(arr.shape),
        'datatype': 'FP32', 'data': arr.flatten().tolist()
    }]}
    r = requests.post(INFER_ENDPOINT, json=payload,
                      headers={'Content-Type': 'application/json'},
                      verify=False, timeout=30)
    r.raise_for_status()
    scores = np.array(r.json()['outputs'][0]['data'])
    exp_s  = np.exp(scores - scores.max())
    return exp_s / exp_s.sum()

fig, axes = plt.subplots(1, len(TEST_IMAGES), figsize=(14, 4))
for ax, (label, img_path) in zip(axes, TEST_IMAGES.items()):
    arr   = preprocess(img_path)
    probs = classify(arr)
    pred  = CLASSES[int(np.argmax(probs))]
    conf  = float(probs.max())
    ax.imshow(Image.open(img_path))
    ax.set_title(f'{label}\n→ {pred} ({conf*100:.0f}%)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/batch-results.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Batch classification complete')
print('   Results saved to /tmp/batch-results.png')